### Pre-processing - Step 1 - Define folder (or URL) where the Python code and the JSON files are located
Provide the folder path of the *dataset* (JSON files) and the folder path of the *codebase* (Python code). 

The paths can be provided to the special UI created below. Do not forget to hit **Submit**.

In [1]:
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

def path_selector(label, global_name):
    """
    Creates a path input widget with validation and stores
    the result in a global variable.
    """
    text = widgets.Text(
        value='',
        placeholder='Enter folder or URL path',
        description=f'{label}:',
        layout=widgets.Layout(width='600px')
    )

    button = widgets.Button(
        description='Submit',
        button_style='primary'
    )

    output = widgets.Output()

    def on_submit(b):
        with output:
            clear_output()
            path = text.value.strip()

            if not path:
                print("⚠️ Please enter a path.")
                return

            if not os.path.exists(path):
                print(f"❌ Path does not exist:\n{path}")
                return

            globals()[global_name] = path
            print(f"✅ {label} path accepted:\n{path}")

    button.on_click(on_submit)

    return widgets.VBox([text, button, output])

# Create two path selectors
json_data_ui = path_selector("Dataset path", "DATASET_PATH")
python_code_ui = path_selector("Python codebase path", "LIBRARY_PATH")

# Display both
display(json_data_ui, python_code_ui)

In [2]:
json_data_folder = str(json_data_ui.children[0].value + "\\")
python_code_path = str(python_code_ui.children[0].value + "\\")

### Pre-processing - Step 2 - Choose the truss typology to load

CHoose the **truss typology** that you want to import from the dataset by using the slider at the bottom of the code block.

In [3]:
# List of truss typology names
truss_typologies = {0: "Pratt", 1: "Howe", 2: "K-Truss", 3: "Warren", 4: "Fink", 5: "All"}

# Create (label, value) pairs: label shows key + name
options = [(f"{key} ({name})", key) for key, name in truss_typologies.items()]

slider = widgets.SelectionSlider(
    options=options,
    value=0,  # default key
    description="Truss type:",
    continuous_update=False
)

# Label widget to show current selection
label = widgets.Label(value=f"Selected: 0 (Pratt)")

def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        selected_key = change['new']
        selected_name = truss_typologies[selected_key]
        print(f"Selected key: {selected_key}, name: {selected_name}")

slider.observe(on_change)

display(slider)

SelectionSlider(continuous_update=False, description='Truss type:', options=(('0 (Pratt)', 0), ('1 (Howe)', 1)…

In [4]:
# Access selected value
truss_typology = truss_typologies[slider.value]
print(f"Selected typology: {truss_typology}")

Selected typology: All


### Pre-processing - Step 3 - Import modules and helpers

In [5]:
import sys
import os
import pickle
sys.path.append(python_code_path)
from HelperFunctions import Utilities
from Model import Graph

### Dataset import
You can import the dataset in two ways: A. From the JSON files, or B. From the pickle files provided.

The former are human readable, but importing them might take 1 hour for each typology and a couple of hours for all of them.

The latter are not human-readable, but importing them takes just a few seconds.

**RECOMMENDATION:** Opt for importing the datapoints from the provided pickle files *(OPTION B)*.

#### OPTION A: Import from JSON files!

In [ ]:
if truss_typology == "All":
    folder_path = json_data_folder + "\\"
    folders = [name for name in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, name))]
    json_graphs = {}
    for folder in folders:
        folder_path = json_data_folder + "\\" + folder    
        json_graphs.update(Utilities.LoadJSONfiles(folder_path, folder))
else:
    folder_path = json_data_folder + "\\" + truss_typology
    json_graphs = Utilities.LoadJSONfiles(folder_path, truss_typology)

print(f"Imported {len(json_graphs)}x JSON files successfully.")

#### OPTION B: Import from pickle files!

In [6]:
with open(json_data_folder + "\\imported_graphs_" + truss_typology + ".pkl", "rb") as file:
    json_graphs = pickle.load(file)
    print("Imported graphs have been deserialized.")

Imported graphs have been deserialized.


### Parsing imported dataset to *Graph* instances (and optionally printing existing attributes)

Unlike the step above, there is no alternative for the step below. The dataset parsing is time-consuming and it can escalate up to 2 or 3 hours if *All* typologies are imported.

In [7]:
graphs = {}
for key_graph in json_graphs:
    G = Graph.ByJSONstring(json_graphs[key_graph])
    graphs[key_graph] = G
    #print(f"Graph {key_graph} has bar elements ranging between {G.MinLength} and {G.MaxLength} units.")

### Extending *Graph* attributes

The code block below illustrates how one can use existing attributes to calculate new metrics and thus extend the provided attributes.

Specifically, the **Static Action** of a truss graph is calculated below.

In [8]:
graph_by_static_action = {}
for key_graph, graph in graphs.items():
    static_action = sum(abs(f) * l for f, l in zip(graph.AxialForces, graph.Lengths))
    graph_by_static_action[key_graph] = static_action

sorted_graphs_by_static_action = dict(sorted(graph_by_static_action.items(), key=lambda x: x[1]))
print(f"Graph {list(sorted_graphs_by_static_action.keys())[0]} has the minimum Static Action, which is equal to: {graph_by_static_action[list(sorted_graphs_by_static_action.keys())[0]]}.")

Graph Warren_p_6_seed_795_graph_1986 has the minimum Static Action, which is equal to: 38.65246632464124.


### Pre-processing - Step 1 - Choose the geometry limits

Provide the numerical constraints of the **bar length** and **inner nodal angle** to be used as filtering criteria below.

In [ ]:
minL = 0.7
minA = 20

### Pre-processing - Step 2 - Assign materiality

Provide materiality to the truss structure so that you can take advantage of the structural performance metrics that are saved as attributes.

As explained in the dataset descriptor, the structural performance metrics are coefficients rather than actual performance metrics.

In [ ]:
percent    = True
disp_units = True
E          = 2.1*10**11 # N/m2
I          = 0.0000251  # m4
A          = 0.004525   # m2
S          = 235*10**6  # N/m2
d          = 7850       # kg/m3
F_m        = 3000       # N/m
L          = 20         # m
F_tot      = F_m*L      # N

### Pre-processing - Step 3 - Choose performance limits

Performance the numerical constraints of the **maximum length**, **maximum allowed displacement** and the **maximum allowed utilisation** to be used as filtering criteria below.

In [ ]:
maxL         = 4.0    # m
allowed_disp = 10     # cm
allowed_util = 90     # ...just a ratio

### Masking - Step 1 - Based on geometric features **(minimum length, minimum angles)**
Apply geometric related filtering criteria

In [ ]:
graph_labels_to_exclude = set()

In [ ]:
for key_graph, graph in graphs.items():
    # minimum length filtering
    min_length = graph.MinLength
    if min_length < minL:
        graph_labels_to_exclude.add(key_graph)
        continue
    
    # minimum angles filtering
    include = True
    angles, _, _, _ = graph.FaceAngles()
    for node_id, node_angles in angles.items():
        if include:
            for angle in node_angles:
                if angle < minA:
                    graph_labels_to_exclude.add(key_graph)
                    include = False
                    break   # exits the inner loop   
        else:
            break

### Masking - Step 2 - Based on performance metrics **(displacement and buckling)**
In this step, the performance related metrics are scaled to respond to the assigned materiality.

In [ ]:
masked_graphs = {}

In [ ]:
for key_graph, grpah in graphs.items():
    if key_graph not in graph_labels_to_exclude:

        # scale the axial forces to the actual load
        min_t_f = graph.MinTensionForce * F_tot
        max_t_f = graph.MaxTensionForce * F_tot
        min_c_f = graph.MinCompressionForce * F_tot
        max_c_f = graph.MaxCompressionForce * F_tot
        
        max_utilization_ = graph.MaxGlobalUtilization(A, S, percent)
        max_utilization = round(max_utilization_ * F_tot, 4)
        max_disp_ = graph.MaxGlobalDisplacement(disp_units)
        max_disp = round(max_disp_ * F_tot / (E*A), 4)

        if max_disp < allowed_disp and graph.MinLength > minL and graph.MaxLength < maxL and max_utilization < allowed_util:
            masked_graphs[key_graph] = graph
        else:
            #print(f'Graph {key_graph} does not meet the required performance criteria.')
            pass
    
    else:
        #print(f'Graph {key_graph} does not meet the required geometric criteria.')
        pass

In [ ]:
print(f"Following your geometric and performance criteria, you have qualified {len(masked_graphs)}x graphs!")

### (Optional) Store/Serialize the qualified datapoints as a pickle file!

In [ ]:
with open(json_data_folder + "\\qualified_graphs_" + truss_typology + ".pkl", "wb") as file:
    pickle.dump(masked_graphs, file)
    print("Qualified graphs have been serialized and saved.")